#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# choose plug in targets or orthogonal targets
plug_in = False

In [ ]:
# set confounders
confounders = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11','x12', 'x13', 'x14', 'x15', 
               'x16', 'x17', 'x18', 'x19', 'x20', 'x21','x22', 'x23', 'x24', 'x25']
input_dim = len(confounders)

#### helpers

In [ ]:
def load_nuisance_models(ns_seed_dir, input_dim, device):
    """Helper to load nuisance models. Set the hidden dim correctly!"""
    paths = {
        "e": ns_seed_dir / "prop_model.pt",
        "m0": ns_seed_dir / "mu0_model.pt",
        "m1": ns_seed_dir / "mu1_model.pt"}

    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing nuisance checkpoint {name}: {path}")

    prop_model = ClassificationHead(input_dim=input_dim, hidden_dim=32).to(device)
    prop_model.load_state_dict(torch.load(paths["e"], map_location=device, weights_only=True))

    m0_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m0_model.load_state_dict(torch.load(paths["m0"], map_location=device, weights_only=True))

    m1_model = RegressionHead(input_dim=input_dim, hidden_dim=128).to(device)
    m1_model.load_state_dict(torch.load(paths["m1"], map_location=device, weights_only=True))

    return prop_model, m0_model, m1_model

In [ ]:
def run_one_trial(train_df, val_df, confounders, device, seed, params, plug_in):
    set_seed(seed)

    # make loaders
    train_loader, _ = make_ranker_loaders(train_df, val_df, confounders, params["kappa"], params["batch_size"])
    _, val_loader = make_cate_loaders(train_df, val_df, confounders, params["batch_size"])

    # init model
    input_dim = len(confounders)
    ranker = ClassificationHead(input_dim, params["hidden_dim"]).to(device)

    # train
    ranker, info = train_ranker(ranker, train_loader, val_loader, device, lr=params['learning_rate'], weight_decay=params['weight_decay'],
                                      max_epochs=params['max_epochs'], patience=params['patience'], seed=seed, fraction_of_pairs=0.1, plug_in=plug_in)

    # store info
    info = {
        **info,
        "kappa": params["kappa"],
        "hidden_dim": params["hidden_dim"],
        "learning_rate": params["learning_rate"],
        "weight_decay": params["weight_decay"],
        "batch_size": params["batch_size"]}

    return ranker, info

In [ ]:
def select_kappa(train_df, val_df, confounders, device, seed, params, plug_in):
    best_model, best_info = None, None

    # loop over kappa values
    for kappa in [0.25, 0.5, 1, 1.5, 3]:
        print(f"kappa - {kappa}")
        trial_params = {**params,"kappa": kappa}

        # train model
        model, info = run_one_trial(train_df, val_df, confounders, device, seed+1, trial_params, plug_in)

        # select by validation loss
        if best_info is None or info["val_autoc"] > best_info["val_autoc"]:
            best_info = info
            best_model = model

    # return  best model
    return best_model, best_info

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/ihdp_rankers.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim=int(row["hidden_dim"]),
    learning_rate=float(row["lr"]),
    weight_decay=float(row["weight_decay"]),
    batch_size=int(row["batch_size"]),
    max_epochs=50,
    patience=5)

In [ ]:
# set directories
out_dir = f'./chkpts/rankers/'
os.makedirs(out_dir, exist_ok=True)

ns_dir = ROOT / 'experiments' / 'supplementary' / 'benchmarks' / 'ihdp' / 'chkpts' / 'nuisances'

In [ ]:
# loop over seeds
for seed in range(50):

    # track progress
    print(f" -> Seed {seed}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data - no two stage splitting
    train_full = pd.read_csv(f'./data/datasets/replication_{seed}/train.csv')
    train_df, val_df = train_test_split(train_full,test_size=0.2,random_state=seed,shuffle=True)

    # load nuisance models
    ns_seed_dir = ns_dir / f"seed_{seed}"
    prop_model, m0_model, m1_model = load_nuisance_models(ns_seed_dir=ns_seed_dir, input_dim=input_dim, device=device)

    # add pseudo outcomes to dataframes
    train_df = compute_dr_scores(train_df, confounders, prop_model, m0_model, m1_model, device)
    val_df = compute_dr_scores(val_df, confounders, prop_model, m0_model, m1_model, device)

    # select kappa per IHDP setting
    ranker, best = select_kappa(train_df=train_df, val_df=val_df, confounders=confounders, device=device, seed=seed,
                                params=params, plug_in=plug_in)

    # checkpoint
    method_name = "plug_in" if plug_in else "orthogonal"
    torch.save(ranker.state_dict(), ckpt_dir / f"{method_name}.pt")